# Private full L2-SP training
Fresh original model; exact prescribed L2-SP objective, seed6304,1250updates/40000examples, microbatch2 accumulation16, FP32 master parameters and CPU AdamW/reference offload. GPU fit/throughput remain unmeasured. Requires original private Task1 input, free T4 and Internet. Preserve raw checkpoint outputs; no duplicate full-weight ZIP.
Intermediate snapshots preserve exact model weights for evaluation, without optimizer state. Final/early-stop checkpoints retain full recovery. Requires18GiB free disk; no existing outputs deleted.


In [ ]:
# Set only after the previous Kaggle draft is stopped and no training is active.
BATCH_HANDOFF_CONFIRMED = False
assert BATCH_HANDOFF_CONFIRMED, 'Stop the prior draft/session, then confirm handoff before Save & Run All'

import shutil, pathlib
free_bytes = shutil.disk_usage('/kaggle/working').free
print('PREFLIGHT_FREE_DISK_GIB',free_bytes / 2**30,flush=True)
assert free_bytes >= 18 * 2**30, 'Need18GiB free for all required model snapshots and final recovery'
meminfo = dict(line.split(':',1) for line in pathlib.Path('/proc/meminfo').read_text().splitlines())
assert int(meminfo['MemAvailable'].split()[0]) * 1024 >= 20 * 2**30, 'Need20GiB available CPU RAM for FP32 offload'


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "c3f4d0692f2987914fb9383951903c09e59efb6e"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)

subprocess.check_call([sys.executable, '-c', "import torch; assert torch.cuda.is_available(), 'Select free T4 GPU'; print('VERIFIED_GPU', torch.__version__, torch.cuda.get_device_name(0), flush=True)"])


In [ ]:
# Verify all original input bytes; tolerate only byte-identical duplicate copies.
import sys, pathlib, zipfile, hashlib
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs, select_original_input
EXPECTED_TRAIN_SHA = '644a46f9ab7a8d368499b2054229cc7324614d97596fbdee1ac3ab5cbb584de9'
archives = list(pathlib.Path('/kaggle/input').rglob('pa3-task1-input.zip'))
if len(archives) == 1:
    input_archive = archives[0]
else:
    original = select_original_input('/kaggle/input', EXPECTED_TRAIN_SHA)
    input_archive = work / 'pa3-task1-input.zip'
    if not input_archive.exists():
        with zipfile.ZipFile(input_archive, 'x', zipfile.ZIP_DEFLATED) as z:
            for name in ('train.jsonl', 'metadata.json', 'source_ids.json'):
                z.write(original / name, 'task1/' + name)
data_dir = unpack_inputs(input_archive, work / 'pa3/data')
assert hashlib.file_digest((data_dir / 'train.jsonl').open('rb'), 'sha256').hexdigest() == EXPECTED_TRAIN_SHA
print('Original training data checksum verified:', data_dir, flush=True)


In [ ]:
import pathlib, subprocess, sys, json, hashlib, shutil, os
run_id = 'l2sp-kaggle-20261102'
run = work / 'pa3/runs' / run_id
assert not run.exists(), 'Independent L2-SP run requires a fresh directory'
# Three weights-only snapshots plus final model/AdamW recovery/export need ~15.6GiB.
# Keep additional headroom; do not create another copy in a final archive.
free = shutil.disk_usage(work).free
assert free >= 18 * 2**30, f'Insufficient checkpoint disk headroom: {free / 2**30:.2f}GiB'
meminfo = dict(line.split(':',1) for line in pathlib.Path('/proc/meminfo').read_text().splitlines())
assert int(meminfo['MemAvailable'].split()[0]) * 1024 >= 20 * 2**30, 'Insufficient available host RAM for FP32 offload'
command = [sys.executable, '-m', 'scripts.kaggle_task1', '--archive', str(input_archive), '--method', 'l2sp', '--memory-strategy', 'cpu-offload', '--compact-intermediates', '--run-id', run_id, '--seconds', '36000', '--max-updates', '1250', '--handoff-verified']
print('START_FULL_L2SP', 'target1250; CPU FP32 AdamW/reference; GPU fit not yet measured', flush=True)
returncode = None
try:
    returncode = subprocess.run(command, cwd=source).returncode
finally:
    status = json.loads((run / 'status.json').read_text()) if (run / 'status.json').is_file() else None
    checkpoints = []
    if run.exists():
        for marker in run.rglob('complete.json'):
            cp = marker.parent
            if (cp / 'recovery.pt').is_file() and (cp / 'summary.json').is_file():
                summary = json.loads((cp / 'summary.json').read_text())
                checkpoints.append((summary['update'], cp, summary))
    latest = max(checkpoints,key=lambda r:r[0]) if checkpoints else None
    finished = bool(returncode == 0 and status and status.get('finished') and latest and latest[0] == 1250 and latest[2]['successful_optimizer_updates'] == 1250 and latest[2]['skipped_updates'] == 0)
    files = {}
    if run.exists():
        for file in sorted(run.rglob('*')):
            if file.is_file():
                with file.open('rb') as stream:
                    digest = hashlib.file_digest(stream,'sha256').hexdigest()
                files[str(file.relative_to(work))] = {'bytes':file.stat().st_size,'sha256':digest}
    evidence = {'returncode':returncode,'status':status,'full_training_verified':finished,'latest_summary':latest[2] if latest else None,'required_markers_present':{str(i):any(r[0]==i for r in checkpoints) for i in (312,625,937,1250)},'raw_output_files':files,'note':'Preserve raw checkpoint directories in saved version outputs. No duplicate weight ZIP; download required files and verify manifest before declaring local preservation.'}
    manifest = work / 'l2sp-checkpoint-manifest.json'
    manifest.write_text(json.dumps(evidence,indent=2))
    print('ACTUAL_L2SP_RESULT',json.dumps({k:v for k,v in evidence.items() if k!='raw_output_files'}),flush=True)
    print('DOWNLOAD_MANIFEST_AND_RAW_CHECKPOINTS',str(manifest),flush=True)
    if not finished: print('INCOMPLETE: preserve actual recoverable outputs; do not claim1250 completion',flush=True)
